# 💎 OpositaIA Fine-Tuning: Mistral 7B + Unsloth (V9 SUPREME GOLD)
## Dataset: Quality Optimized (Cohere Rerank) | Target: 1 Epoch

Este notebook entrena el **MASTER_DATASET_v9_GOLD_OPTIMIZED.jsonl**.

**Pasos Previos Obligatorios**:
1. Sube `MASTER_DATASET_v9_GOLD_OPTIMIZED.jsonl` a la raíz de tu Google Drive.
2. Ejecuta este notebook celda a celda.

In [ ]:
# 0. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/OpositaIA_Models/Mistral_v9_Gold_Checkpoint"

In [ ]:
%%capture
# 1. Instalar Unsloth y dependencias
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Configurar Modelo (Mistral v0.3 4-bit)
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
# 3. Cargar Dataset V9 GOLD
from datasets import load_dataset

# RUTA CRÍTICA
dataset_path = "/content/drive/MyDrive/MASTER_DATASET_v9_GOLD_OPTIMIZED.jsonl"

alpaca_prompt = """{instruction}\n\n{input}\n\n{output}"""

def formatting_prompts_func(examples):
    instructions = examples["question"]
    outputs      = examples["answer"]
    texts = []
    for i, out in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction=i, input="", output=out) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files = dataset_path, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Entrenar (1 Época Completa)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = output_dir,
        save_strategy = "steps",
        save_steps = 100, 
        save_total_limit = 2,
    ),
)

trainer.train()

In [ ]:
# 5. Exportar a GGUF (q4_k_m)
model.save_pretrained_gguf("opositaia_mistral_v9_gguf", tokenizer, quantization_method = "q4_k_m")

# Guardar en Drive
!cp opositaia_mistral_v9_gguf-unsloth.gguf /content/drive/MyDrive/OpositaIA_Models/mistral_7b_v9_supreme_final.gguf